In [ ]:
%load_ext watermark


In [ ]:
import itertools as it
import os

import matplotlib as mpl
from matplotlib import pyplot as plt
import pandas as pd
from phyloframe import _auxlib as pfa
from phyloframe import legacy as pfl
from pyfonts import load_google_font
import seaborn as sns
from teeplot import teeplot as tp

import pylib


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = os.environ.get("NOTEBOOK_NAME", "2026-03-04-fossil-lineage")
teeplot_subdir


In [ ]:
pfa.seed_random(1)


In [ ]:
font = load_google_font("Merriweather", weight=300)
mpl.font_manager.fontManager.addfont(font.get_file())
plt.rcParams["font.family"] = font.get_name()


## Prep Data


In [ ]:
df_pure = pfl.alifestd_join_roots(
    pd.read_parquet("https://osf.io/pfvsg/download"),
)
df_pure


In [ ]:
df_sweep = pfl.alifestd_join_roots(
    pd.read_parquet("https://osf.io/nk69s/download"),
)
df_sweep


In [ ]:
dfs = []
for df in (df_pure, df_sweep):
    df["x"] = df["position"] // df["nCol"]
    df["x_"] = df["x"] / df["nRow"]
    df["y"] = df["position"] % df["nCol"]
    df["y_"] = df["y"] / df["nCol"]

    df["origin_time"] = df["dstream_rank"]

    dfs.append(df)

df_pure, df_sweep = dfs


## Plot Lineage Tree


In [ ]:
for regime, criterion, seed in it.product(
    ("pure", "sweep"),
    ("x", "y", "layer"),
    range(5),
):
    df = {
        "pure": df_pure,
        "sweep": df_sweep,
    }[regime].copy()
    df["criterion"] = df["layer"] * df[criterion].max() + df[criterion] // 100
    df = pfl.alifestd_downsample_tips_lineage_stratified_asexual(
        df,
        criterion_stratify="layer",
        criterion_target="criterion",
        seed=seed,
    )
    df = pfl.alifestd_to_working_format(df)
    df = pfl.alifestd_downsample_tips_lineage_asexual(
        df,
        criterion_target="layer",
        num_tips=50,
        seed=seed,
    )

    df = pfl.alifestd_collapse_unifurcations(df)
    df = pfl.alifestd_delete_unifurcating_roots_asexual(df)
    df["origin_time"] -= df["origin_time"].min()
    df = pfl.alifestd_mark_node_depth_asexual(df)

    for layout in "vertical", "horizontal", "radial":

        with tp.teed(
            pylib.chloropleth.draw_ctree,
            df,
            x="x_",
            y="y_",
            cmap=pylib.cmap.bcyr.get_color,
            layout=layout,
            scatter_kws=dict(
                edgecolor="none",
                s=4,
            ),
            scatter_shuffle=1,
            tree_kws=dict(
                edge=dict(
                    color="gainsboro",
                    linewidth=0.5,
                ),
                margins=-0.05,
            ),
            teeplot_outattrs={"regime": regime, "seed": seed},
            teeplot_subdir=teeplot_subdir,
        ) as teed:
            teed.invert_yaxis()
            teed.figure.set_size_inches(3, 1.33)

    data = df.copy().sort_values(by="node_depth").dropna(subset=["x_", "y_"])
    data = data[data["is_leaf"]].copy()
    data = data.reset_index(drop=True).copy()
    with tp.teed(
        pylib.chloropleth.draw_cscatter,
        data.copy(),
        x="x",
        y="y",
        cmap=pylib.cmap.bcyr.get_color,
        despine=True,
        major=100,
        minor=None,
        xmax=1170,
        ymax=755,
        scatter_kws=dict(
            alpha=0.7,
            color="none",
            edgecolor="white",
            linewidth=1,
            linestyle="-",
            marker="o",
            s=10,
            zorder=120,
        ),
        teeplot_outattrs={"cmap": "bcyr", "regime": regime, "seed": seed},
        teeplot_subdir=teeplot_subdir,
    ) as teed:
        for i in range(len(data)):
            teed.plot(
                data["x"][: i + 1],
                data["y"][: i + 1],
                alpha=1 / len(data),
                color="black",
                lw=6,
            )
        teed.plot(
            data["x"],
            data["y"],
            alpha=0.2,
            color="black",
            lw=6,
            zorder=150,
        )
        teed.set_xlabel(None)
        teed.set_ylabel(None)
        teed.set_aspect("equal")
        fig = teed.figure
        fig.set_size_inches(3, 2)
        fig.tight_layout()
